# EXPT NO: 9
## Transformer-Based LLM Chatbot Using Streamlit

**Name:** SWATHI V.  
**Roll No:** 24BAD122  
**Department:** AI & Data Science

**Aim:** To develop an interactive chatbot using a pre-trained Transformer-based LLM and deploy it using Streamlit.

**Model used:** `distilgpt2` from Hugging Face. It is a lightweight model suitable for demonstrating text generation. Its responses may be less accurate than larger instruction-tuned chat models.

## Part A – Loading and Configuring a Pre-trained LLM

Run the following cells in Google Colab or Jupyter Notebook. The first cell installs the required libraries.

In [1]:
!pip install -q transformers torch streamlit

In [2]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_NAME = "distilgpt2"

# Load the tokenizer and pre-trained Transformer language model
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)

# GPT-2 models do not define a padding token by default
tokenizer.pad_token = tokenizer.eos_token
model.config.pad_token_id = tokenizer.eos_token_id

device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)

print("Model loaded successfully!")
print("Running on:", device)

config.json:   0%|          | 0.00/762 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  353MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Model loaded successfully!
Running on: cpu


In [3]:
# Tokenize a sample user prompt
sample_prompt = "Artificial intelligence is"

inputs = tokenizer(sample_prompt, return_tensors="pt").to(device)

print("Input tokens:", inputs["input_ids"])
print("Token count:", inputs["input_ids"].shape[1])

Input tokens: tensor([[8001, 9542, 4430,  318]])
Token count: 4


In [4]:
# Generate a response using temperature and maximum output length
with torch.no_grad():
    output_ids = model.generate(
        **inputs,
        max_new_tokens=60,
        temperature=0.7,
        do_sample=True,
        top_p=0.9,
        pad_token_id=tokenizer.eos_token_id
    )

generated_text = tokenizer.decode(output_ids[0], skip_special_tokens=True)
print("Generated response:\n", generated_text)

Generated response:
 Artificial intelligence is an art that has a lot of appeal, and it is able to be used to develop, refine, and improve. However, the problem is that it is not yet possible to build a machine that can perform a very specific task. The most important thing is to understand that a machine that can perform


## Part B – Implementing Chatbot Functionality

The function below accepts a user query, includes recent conversation history, and generates a response. Conversation history helps the model retain context during the current session.

In [5]:
def generate_response(user_query, chat_history=None, max_new_tokens=100, temperature=0.7):
    if chat_history is None:
        chat_history = []

    # Build a prompt using recent conversation turns
    prompt_parts = []
    for user_text, bot_text in chat_history[-3:]:
        prompt_parts.append(f"User: {user_text}\nAssistant: {bot_text}")

    prompt_parts.append(f"User: {user_query}\nAssistant:")
    prompt = "\n".join(prompt_parts)

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=700
    ).to(device)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=max(temperature, 0.01),
            do_sample=True,
            top_p=0.9,
            repetition_penalty=1.1,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id
        )

    # Decode only the newly generated tokens
    new_tokens = output_ids[0][inputs["input_ids"].shape[1]:]
    answer = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

    if not answer:
        answer = "I could not generate a response. Please try another question."

    return answer

In [6]:
# Test the chatbot with different queries
history = []

questions = [
    "What is machine learning?",
    "Why are Transformers useful in language models?",
    "Give one application of artificial intelligence."
]

for question in questions:
    answer = generate_response(question, history)
    print("User:", question)
    print("Chatbot:", answer)
    print("-" * 60)
    history.append((question, answer))

User: What is machine learning?
Chatbot: Machine Learning and AI. I’m a PhD student in Computer Science from Cornell University, where my dissertation research focuses on the use of intelligent systems to help solve complex problems such as computer programs or software engineering (e.g., programming languages). In this thesis, I am interested primarily about using artificial intelligence techniques for real-world problem solving solutions that can be implemented by any user who has an understanding of computational modeling skills with no experience with Artificial Intelligence. The topics covered include how we model human
------------------------------------------------------------
User: Why are Transformers useful in language models?
Chatbot: We have already created many great tools like Google Translate; our Python program was used extensively over time so it will allow you to get better at handling large numbers of users' data within your system without having to worry about "the

## Part C – Developing an Interactive Streamlit Interface

Run this cell to create the Streamlit application file named `app.py`. The app uses the same Hugging Face model and displays messages in a chat-style layout.

In [7]:
streamlit_code = r"""
import streamlit as st
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_NAME = "distilgpt2"

@st.cache_resource
def load_model():
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)
    tokenizer.pad_token = tokenizer.eos_token
    model.config.pad_token_id = tokenizer.eos_token_id
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model.to(device)
    return tokenizer, model, device

tokenizer, model, device = load_model()

def generate_reply(question, history):
    prompt_lines = []
    for item in history[-3:]:
        prompt_lines.append(f"User: {item['content']}" if item["role"] == "user"
                            else f"Assistant: {item['content']}")
    prompt_lines.append(f"User: {question}\\nAssistant:")
    prompt = "\\n".join(prompt_lines)

    inputs = tokenizer(prompt, return_tensors="pt",
                       truncation=True, max_length=700).to(device)

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=100,
            temperature=0.7,
            do_sample=True,
            top_p=0.9,
            repetition_penalty=1.1,
            pad_token_id=tokenizer.eos_token_id
        )

    new_tokens = output[0][inputs["input_ids"].shape[1]:]
    answer = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
    return answer or "Please try asking in a different way."

st.set_page_config(page_title="Transformer LLM Chatbot", page_icon="🤖")
st.title("🤖 Transformer-Based LLM Chatbot")
st.caption("Powered by Hugging Face Transformers and DistilGPT-2")

if "messages" not in st.session_state:
    st.session_state.messages = []

for message in st.session_state.messages:
    with st.chat_message(message["role"]):
        st.markdown(message["content"])

user_prompt = st.chat_input("Ask me a question...")
if user_prompt:
    st.session_state.messages.append({"role": "user", "content": user_prompt})
    with st.chat_message("user"):
        st.markdown(user_prompt)

    with st.chat_message("assistant"):
        with st.spinner("Generating response..."):
            response = generate_reply(user_prompt, st.session_state.messages[:-1])
            st.markdown(response)

    st.session_state.messages.append({"role": "assistant", "content": response})

if st.sidebar.button("Clear conversation"):
    st.session_state.messages = []
    st.rerun()
"""

with open("app.py", "w", encoding="utf-8") as file:
    file.write(streamlit_code)

print("app.py created successfully!")

app.py created successfully!


## Part D – Application Testing and Deployment

### 1. Run the Streamlit application

Open a terminal in the folder containing `app.py` and execute:

```bash
streamlit run app.py
```

In Google Colab, Streamlit needs a public tunnel or a deployment platform to be accessed through a browser. The simplest option for this practical is to download the generated files and run them locally.

### 2. Create requirements.txt

Run the next cell to create the dependency file.

In [8]:
requirements = """streamlit
transformers
torch
"""

with open("requirements.txt", "w", encoding="utf-8") as file:
    file.write(requirements)

print("requirements.txt created successfully!")

requirements.txt created successfully!


### 3. GitHub and Hugging Face Spaces deployment

1. Create a GitHub repository.
2. Upload `app.py` and `requirements.txt`.
3. Create a new **Streamlit** Space on Hugging Face Spaces.
4. Upload the same files or connect the GitHub repository.
5. Wait for the application to build, then open the Space URL to test it.

### Testing observations

| Test | Expected observation |
|---|---|
| Enter a simple question | The chatbot generates text |
| Ask follow-up questions | Recent conversation turns are included in the prompt |
| Clear conversation | The displayed chat history is removed |
| Ask a complex factual question | The model may produce inaccurate or incomplete information |

**Limitations:** The model may hallucinate, generate repetitive text, respond slowly on CPU, and may not follow instructions reliably. Do not treat its output as verified information.